# BeeNoise — Colab demo

Runs the full pipeline (denoise -> diarize -> speaker ID -> transcribe -> merge) on Google Colab's free compute, and launches the project's web UI with a public link you can share for the length of this session.

**Before you run anything:**
1. `Runtime` -> `Change runtime type` -> select a **T4 GPU** (free tier). Works on CPU too, just noticeably slower.
2. `Runtime` -> `Run all`, then follow the two prompts partway down (pasting your HuggingFace token).

**This is a temporary environment.** Everything here — the cloned repo, any speakers you enroll, any files you process — disappears when this Colab session disconnects or times out. It's for demos, not permanent hosting. See [QUICKSTART.md](https://github.com/gretil3/BeeNoise/blob/main/QUICKSTART.md) for running it for real on your own machine.

## 0. Check what compute you got

In [ ]:
!nvidia-smi || echo 'No GPU detected — CPU only. Runtime > Change runtime type > T4 GPU for a big speed-up.'

## 1. Set up the environment (~3-5 min)

Pins Python 3.10 in its own virtualenv, independent of whatever Python version Colab's base image happens to ship — one of this project's dependencies (DeepFilterNet) has no installable build for 3.12+.

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y software-properties-common > /dev/null
add-apt-repository -y ppa:deadsnakes/ppa > /dev/null 2>&1
apt-get -qq update
apt-get -qq install -y python3.10 python3.10-venv python3.10-dev > /dev/null

rm -rf BeeNoise
git clone --quiet https://github.com/gretil3/BeeNoise
cd BeeNoise
python3.10 -m venv .venv
.venv/bin/pip install -q --upgrade pip
echo 'Virtualenv ready.'

In [ ]:
%%bash
set -e
cd BeeNoise
if nvidia-smi > /dev/null 2>&1; then
  echo 'Installing GPU (CUDA) torch...'
  .venv/bin/pip install -q torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
else
  echo 'Installing CPU torch...'
  .venv/bin/pip install -q torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cpu
fi
.venv/bin/pip install -q -r requirements.txt
echo 'Dependencies installed.'

## 2. HuggingFace token (needed for diarization)

1. Create a free account at [huggingface.co](https://huggingface.co), then make a **Read** token: `Settings -> Access Tokens -> New token`.
2. Accept the terms on **both** of these pages, logged in as that same account (each has an "Agree and access repository" button — for the required "website" field you can use `https://github.com/gretil3/BeeNoise`):
   - https://huggingface.co/pyannote/speaker-diarization-3.1
   - https://huggingface.co/pyannote/segmentation-3.0
3. Run the next cell and paste the token when prompted — it's a hidden input, never shown or saved in this notebook file.

In [ ]:
import pathlib
from getpass import getpass

token = getpass('Paste your HuggingFace token (hidden): ').strip()
pathlib.Path('BeeNoise/.env').write_text('HF_TOKEN=' + token + chr(10))
print('Saved.')

## 3. Verify everything (downloads ~1-2GB of models, one time)

In [ ]:
!cd BeeNoise && .venv/bin/python -m src.check_env --models

## 4. Use the GPU, if you have one

`config.yaml` defaults every model to CPU. If a GPU was detected in step 0, switch diarization and transcription over to it — this alone is usually the difference between a clip taking minutes vs. seconds.

In [ ]:
import pathlib
import re
import shutil
import subprocess

has_gpu = (shutil.which('nvidia-smi') is not None
           and subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0)
cfg_path = pathlib.Path('BeeNoise/config.yaml')
text = cfg_path.read_text()

if has_gpu:
    new = re.sub(r'(?m)^(\s*device:\s*)cpu\b', r'\1cuda', text)
    new = re.sub(r'(?m)^(\s*compute_type:\s*)int8\b', r'\1float16', new)
    if new == text:
        print('GPU detected, but config.yaml had nothing to switch (already on GPU, or its format changed).')
    else:
        cfg_path.write_text(new)
        print('GPU detected — diarization and transcription set to use it.')
else:
    print('No GPU — staying on CPU. Everything still works, just slower.')

## 5. Launch the app

Starts the same [Gradio](https://gradio.app) UI you'd run locally with `python -m src.demo_ui`, but with `--share` so it prints a public `https://....gradio.live` link — open that link (on any device) for a browser UI with tabs to enroll a speaker and to upload a recording and get denoised audio, speaker-labelled subtitles, and a subtitled video back.

This cell finishes once the link appears; the server keeps running in the background for as long as this Colab session stays connected. The link stops working when the session ends. Safe to re-run this cell any time (e.g. after pulling an update) — it stops any previous instance first.

In [ ]:
%%bash
cd BeeNoise
pkill -f 'src.demo_ui' || true
sleep 1
nohup .venv/bin/python -u -m src.demo_ui --share > demo.log 2>&1 &
echo 'Starting the app — first run can take a few minutes (loading torch, gradio, etc.)...'
for i in $(seq 1 150); do
  if grep -q 'Running on public URL' demo.log; then
    grep 'Running on public URL' demo.log
    exit 0
  fi
  sleep 2
done
echo 'Still starting after 5 minutes — run the next cell to see the log and check for a real error.'

In [ ]:
# If the link didn't show up above, check what happened:
!tail -n 40 BeeNoise/demo.log

---
### Stop the app
Run this to shut the server down (e.g. before re-running step 5).

In [ ]:
!pkill -f 'src.demo_ui' || true
print('Stopped (if it was running).')